In [ ]:
#fix weather table for RNA analysis - PATHOCOM

#input we have: 
#google drive from weather underground for all sites 
#daily values for 30 days before sampling + sampling day + 1 week after sampling

#RNA samples were only from Fall 2022 and Spring 2023

#output we want: 
#For TEMPERATURE and HUMIDITY: 3 days before sampling + sampling day
### Mean average temperature and humidity: Sum(tempAvg)/4 days
### Average temperature fluctuation: Sum(tempHigh - tempLow)/4 → we do not do a range for humidity, the average is enough

#For PRECIPITATION (precipTotal): a sum (NOT AVERAGE) of the total
#precipitation over 30 days prior to sampling + the sampling day

In [ ]:
library(readxl)
library(purrr)
library(tidyverse)
library(dplyr)
library(ggplot2)
library(lubridate)

In [ ]:
path <- "weatherundergound_results.xlsx"
sheets <- excel_sheets(path)

head(sheets) # one tab per country


In [ ]:
weather_raw <- map_df(sheets, ~read_excel(path, sheet =.x) %>%
                        mutate(country = .x))


In [ ]:
head(weather_raw)

In [ ]:
weather_subset <- weather_raw %>%
  mutate(
    season = case_when(
      season %in% c("2022-Autumn", "Fall2022") ~ "2022-Autumn", 
      season %in% c("2023-Spring", "Spring2023") ~ "2023-Spring", 
      TRUE ~ season)) %>%
  select(country, season, locality, site_lat, site_lon, collection_date, station_id, tz, obsTimeLocal, humidityAvg, tempAvg, tempHigh, tempLow, precipTotal)%>%
  glimpse()

In [ ]:
#normalize dates

weather <- weather_subset %>%
  mutate(
    obsDate = as.Date(obsTimeLocal), 
    collection_date= as.Date(collection_date)
  )

tail(weather)

In [ ]:
# keep only from Nov/Dec 2022 and Feb/Mar/Apr 2023

weather_kept <- weather %>%
  filter(
    (year (collection_date) == 2022 & month(collection_date) %in% c(11, 12)) |
    (year (collection_date) == 2023 & month (collection_date) %in% c(02:04))  
  )

glimpse(weather_kept)

ggplot(weather_kept, aes(x = obsDate, y = tempAvg, group = season, fill= season))+
  geom_boxplot()+
  facet_wrap(~ country, scales = "free_x")

In [ ]:
#mean temperature over 3d prior + collection_date

weather_summary <- weather_kept %>%
  mutate(
    obsDate        = as.Date(obsTimeLocal),
    collection_date = as.Date(collection_date)
  ) %>%
  group_by(country, locality, season, collection_date, site_lat, site_lon) %>%
  summarise(
    # ---- Temperature & humidity: 3 days before + collection day ----
    mean_temp_4d = mean(
      tempAvg[obsDate >= (collection_date - days(3)) & obsDate <= collection_date],
      na.rm = TRUE
    ),
    mean_hum_4d = mean(
      humidityAvg[obsDate >= (collection_date - days(3)) & obsDate <= collection_date],
      na.rm = TRUE
    ),
    mean_fluctuation_4d = mean(
      (tempHigh - tempLow)[obsDate >= (collection_date - days(3)) & obsDate <= collection_date],
      na.rm = TRUE
    ),

    # ---- Precipitation: 30 days before + collection day ----
    precip_30d_sum = sum(
      precipTotal[obsDate >= (collection_date - days(30)) & obsDate <= collection_date],
      na.rm = TRUE
    ),

    # diagnostics
    n_temp_days = sum(obsDate >= (collection_date - days(3))  & obsDate <= collection_date),
    n_precip_days = sum(obsDate >= (collection_date - days(30)) & obsDate <= collection_date),

    .groups = "drop"
  )


In [ ]:
head(weather_summary)

In [ ]:
abiotic_cols <- c("mean_temp_4d", "mean_hum_4d",
                  "mean_fluctuation_4d", "precip_30d_sum")

# 1) locality × season means
weather_imputed <- weather_summary %>%
  group_by(country, locality, season) %>%
  mutate(
    across(
      all_of(abiotic_cols),
      ~ ifelse(is.na(.x), mean(.x, na.rm = TRUE), .x)
    )
  ) %>%
  ungroup()

# 2) still NA? use country × season mean as fallback
weather_imputed <- weather_imputed %>%
  group_by(country, season) %>%
  mutate(
    across(
      all_of(abiotic_cols),
      ~ ifelse(is.na(.x), mean(.x, na.rm = TRUE), .x)
    )
  ) %>%
  ungroup()

# check remaining NAs
colSums(is.na(weather_imputed[abiotic_cols]))


In [ ]:
abiotic_cols <- c("mean_temp_4d", "mean_hum_4d",
                  "mean_fluctuation_4d", "precip_30d_sum")

weather_imputed <- weather_summary %>%
  group_by(country, locality, season) %>%
  mutate(across(all_of(abiotic_cols),
                ~ ifelse(is.na(.x), mean(.x, na.rm = TRUE), .x))) %>%
  ungroup() %>%
  group_by(country, season) %>%
  mutate(across(all_of(abiotic_cols),
                ~ ifelse(is.na(.x), mean(.x, na.rm = TRUE), .x))) %>%
  ungroup()


In [ ]:
colSums(is.na(weather_imputed[abiotic_cols]))

In [ ]:
write.csv(weather_summary, "251028_WeatherUnderground_Summary_Complete.csv")

In [ ]:
glimpse(weather_summary)

In [ ]:
# ---------- GLOBAL transforms for dual axis ----------
rng <- summarise(weather_summary,
                 tmin = min(mean_temp_4d, na.rm = TRUE),
                 tmax = max(mean_temp_4d, na.rm = TRUE),
                 hmin = min(mean_hum_4d, na.rm = TRUE),
                 hmax = max(mean_hum_4d, na.rm = TRUE),
                 pmax = max(precip_30d_sum, na.rm = TRUE)
)

f <- (rng$tmax - rng$tmin) / (rng$hmax - rng$hmin)         # humidity -> temp scale (slope)
a <- rng$tmin - f * rng$hmin                                # humidity -> temp scale (intercept)

# Scale precip into temp-axis space (0..1 of max precip -> temp range)
precip_df <- weather_summary %>%
  group_by(country, season) %>%
  summarise(precip_30d_mean = mean(precip_30d_sum, na.rm = TRUE), .groups = "drop") %>%
  mutate(precip_scaled = (precip_30d_mean / rng$pmax) * (rng$tmax - rng$tmin) + rng$tmin)

# Build long frame for boxplots (temp as-is; humidity transformed)

plot_df <- bind_rows(
  weather_summary %>% transmute(country, season, metric = "Temperature (°C)", value_plot = mean_temp_4d),
  weather_summary %>% transmute(country, season, metric = "Humidity (%)",     value_plot = a + f*mean_hum_4d)
)

# Colors
cols <- c("Temperature (°C)" = "#2A9D8F",   # teal
          "Humidity (%)"     = "#E76F51")   # coral



p_top <- ggplot(plot_df, aes(x = season, y = value_plot, fill = metric)) +
  #geom_violin()+
  geom_boxplot(width = 0.55, position = position_dodge(width = 0.78), outlier.alpha = 0.25) +
  stat_summary(aes(group = metric), fun = median, geom = "point",
               position = position_dodge(width = 0.78), size = 2.2, color = "black") +
  facet_wrap(~ country, nrow = 1) +
  scale_y_continuous(
    name = "Temperature (°C)",
    sec.axis = sec_axis(~ (. - a) / f, name = "Humidity (%)"),
    expand = expansion(mult = c(0.02, 0.08))
  ) +
  scale_fill_manual(values = c("Temperature (°C)"="#2A9D8F","Humidity (%)"="#E76F51"), name = NULL) +
  labs(title = "Temperature & Humidity (4-day window)") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "top", strip.text = element_text(face = "bold"))

# --- Bottom: Precipitation bars in REAL units (mm) ---
precip_df <- weather_summary %>%
  group_by(country, season) %>%
  summarise(precip_30d_sum_mean = mean(precip_30d_sum, na.rm = TRUE), .groups = "drop")

p_bottom <- ggplot(precip_df, aes(x = season, y = precip_30d_sum_mean, fill = country)) +
  geom_col(width = 0.55, position = position_dodge(width = 0.7)) +
  geom_text(aes(label = round(precip_30d_sum_mean, 1)),
            position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  facet_wrap(~ country, nrow = 1) +
  labs(title = "Precipitation (30-day total up to collection day)",
       x = "Season", y = "Precipitation (mm)") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "none")

p_top
p_bottom